# 🎙️ Hindi ASR — Whisper Fine-Tuning & WER Evaluation

**Module 1 (Q1)**: Fine-tune `openai/whisper-small` on Hindi ASR data and evaluate on FLEURS Hindi test set.

This notebook is designed to run on **Google Colab with GPU runtime**.

---
## Setup Checklist
1. **Runtime → Change runtime type → GPU (T4 or better)**
2. **Add your HuggingFace token** in the cell below
3. **Run all cells** sequentially

## 📦 Step 0: Install Dependencies

In [3]:
'''!pip install -q torch transformers datasets accelerate jiwer librosa soundfile pydub tqdm rapidfuzz python-dotenv''' 

!pip install -q transformers==4.44.2 datasets==2.21.0 accelerate huggingface-hub jiwer librosa soundfile pydub tqdm

## 🔑 Step 1: Set API Keys

Your HuggingFace token is needed to download the Whisper model and FLEURS dataset.
Get one from: https://huggingface.co/settings/tokens

In [4]:
import os

# ─── PUT YOUR HUGGINGFACE TOKEN HERE ──────────────────────────────
HF_TOKEN = "your_huggingface_token_here"  # <-- REPLACE THIS
# ──────────────────────────────────────────────────────────────────

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

# Login to HuggingFace
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print("✅ Logged in to HuggingFace")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ Logged in to HuggingFace


## 🖥️ Step 2: Verify GPU

In [5]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {gpu_name} ({gpu_mem:.1f} GB)")
    print(f"   CUDA Version: {torch.version.cuda}")
    print(f"   PyTorch: {torch.__version__}")
else:
    print("⚠️ No GPU detected! Go to Runtime → Change runtime type → GPU")
    print("   Training will be VERY slow on CPU.")

✅ GPU: Tesla T4 (15.6 GB)
   CUDA Version: 12.8
   PyTorch: 2.10.0+cu128


## ⚙️ Step 3: Configuration

In [6]:
import os
import unicodedata
import re

# ─── Project Configuration ────────────────────────────────────────
MODEL_NAME = "openai/whisper-small"
LANGUAGE = "hi"
TASK = "transcribe"
SAMPLE_RATE = 16000

# FLEURS test set for evaluation
FLEURS_DATASET = "google/fleurs"
FLEURS_LANG_CODE = "hi_in"
FLEURS_SPLIT = "test"

# Training hyperparameters
TRAINING_CONFIG = {
    "learning_rate": 1e-5,
    "warmup_steps": 500,
    "num_train_epochs": 3,
    "per_device_train_batch_size": 8,
    "per_device_eval_batch_size": 8,
    "gradient_accumulation_steps": 2,
    "fp16": True,
    "eval_steps": 500,
    "save_steps": 500,
    "logging_steps": 50,
    "save_total_limit": 3,
    "load_best_model_at_end": True,
    "metric_for_best_model": "wer",
    "greater_is_better": False,
}

# Output directory
OUTPUT_DIR = "./whisper_hindi_finetuned"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"✅ Config ready")
print(f"   Model: {MODEL_NAME}")
print(f"   LR: {TRAINING_CONFIG['learning_rate']}")
print(f"   Epochs: {TRAINING_CONFIG['num_train_epochs']}")
print(f"   Batch: {TRAINING_CONFIG['per_device_train_batch_size']} × {TRAINING_CONFIG['gradient_accumulation_steps']} accum = {TRAINING_CONFIG['per_device_train_batch_size'] * TRAINING_CONFIG['gradient_accumulation_steps']} effective")

✅ Config ready
   Model: openai/whisper-small
   LR: 1e-05
   Epochs: 3
   Batch: 8 × 2 accum = 16 effective


## 🛠️ Step 4: Helper Functions (Text Normalization)

In [7]:
def normalize_unicode(text):
    """NFC normalization — ensures consistent Devanagari encoding."""
    return unicodedata.normalize("NFC", text) if text else ""

def clean_text(text):
    """Basic Hindi text cleaning."""
    if not text:
        return ""
    text = normalize_unicode(text)
    text = re.sub(r'[।॥,;:!?\'\"\.\-\(\)\[\]{}]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def normalize_for_wer(text):
    """Aggressive normalization for WER comparison."""
    if not text:
        return ""
    text = normalize_unicode(text)
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("✅ Text utilities loaded")
print(f"   Test: normalize_for_wer('नमस्ते। 123') = '{normalize_for_wer('नमस्ते। 123')}'")

✅ Text utilities loaded
   Test: normalize_for_wer('नमस्ते। 123') = 'नमसत'


## 📥 Step 5: Load Model & Processor

In [8]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor

print(f"Loading {MODEL_NAME}...")

processor = WhisperProcessor.from_pretrained(
    MODEL_NAME,
    language=LANGUAGE,
    task=TASK,
)

model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

# Hindi-specific configuration
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language=LANGUAGE, task=TASK
)
model.config.suppress_tokens = []
model.config.use_cache = False  # for gradient checkpointing

num_params = sum(p.numel() for p in model.parameters())
print(f"✅ Model loaded: {num_params:,} parameters")
print(f"   Forced decoder IDs set for Hindi transcription")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Loading openai/whisper-small...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


✅ Model loaded: 241,734,912 parameters
   Forced decoder IDs set for Hindi transcription


## 📊 Step 6: Load FLEURS Hindi Dataset

FLEURS (Few-shot Learning Evaluation of Universal Representations of Speech) provides
standardized test data for Hindi ASR evaluation.

In [9]:
'''
from datasets import load_dataset, Audio

print(f"Loading FLEURS {FLEURS_LANG_CODE}...")

# Load train+validation for fine-tuning, test for evaluation
fleurs_train = load_dataset(FLEURS_DATASET, FLEURS_LANG_CODE, split="train")
fleurs_val = load_dataset(FLEURS_DATASET, FLEURS_LANG_CODE, split="validation")
fleurs_test = load_dataset(FLEURS_DATASET, FLEURS_LANG_CODE, split="test")

# Ensure audio is at 16kHz
fleurs_train = fleurs_train.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))
fleurs_val = fleurs_val.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))
fleurs_test = fleurs_test.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))

print(f"✅ FLEURS Hindi loaded:")
print(f"   Train: {len(fleurs_train)} samples")
print(f"   Validation: {len(fleurs_val)} samples")
print(f"   Test: {len(fleurs_test)} samples")

'''

from datasets import load_dataset, Audio

print(f"Loading FLEURS {FLEURS_LANG_CODE}...")

fleurs_train = load_dataset(FLEURS_DATASET, FLEURS_LANG_CODE, split="train", trust_remote_code=True)
fleurs_val = load_dataset(FLEURS_DATASET, FLEURS_LANG_CODE, split="validation", trust_remote_code=True)
fleurs_test = load_dataset(FLEURS_DATASET, FLEURS_LANG_CODE, split="test", trust_remote_code=True)

fleurs_train = fleurs_train.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))
fleurs_val = fleurs_val.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))
fleurs_test = fleurs_test.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))

print(f"✅ FLEURS Hindi loaded:")
print(f"   Train: {len(fleurs_train)} samples")
print(f"   Validation: {len(fleurs_val)} samples")
print(f"   Test: {len(fleurs_test)} samples")

Loading FLEURS hi_in...


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

✅ FLEURS Hindi loaded:
   Train: 2120 samples
   Validation: 239 samples
   Test: 418 samples


## 🔧 Step 7: Prepare Dataset for Training

In [10]:
def prepare_dataset(batch):
    """Process a single FLEURS example for Whisper training."""
    audio = batch["audio"]
    
    # Extract log-mel features
    batch["input_features"] = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
        return_tensors="pt",
    ).input_features[0]
    
    # Tokenize transcription
    batch["labels"] = processor.tokenizer(batch["transcription"]).input_ids
    return batch

print("Preparing training data...")
train_dataset = fleurs_train.map(
    prepare_dataset,
    remove_columns=fleurs_train.column_names,
    num_proc=1,
)

print("Preparing validation data...")
eval_dataset = fleurs_val.map(
    prepare_dataset,
    remove_columns=fleurs_val.column_names,
    num_proc=1,
)

print(f"✅ Datasets prepared:")
print(f"   Train: {len(train_dataset)} samples")
print(f"   Eval: {len(eval_dataset)} samples")

Preparing training data...


Map:   0%|          | 0/2120 [00:00<?, ? examples/s]

Preparing validation data...


Map:   0%|          | 0/239 [00:00<?, ? examples/s]

✅ Datasets prepared:
   Train: 2120 samples
   Eval: 239 samples


## 📦 Step 8: Data Collator

In [11]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class WhisperDataCollator:
    """
    Packages variable-length audio features and labels into uniform batches.
    Replaces padding tokens with -100 so loss ignores them.
    """
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Pad audio features
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # Pad label sequences
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # Replace padding with -100 for loss masking
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        # Remove BOS if present
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = WhisperDataCollator(processor=processor)
print("✅ Data collator ready")

✅ Data collator ready


## 📊 Step 9: Evaluation Metrics

In [12]:
import jiwer
from functools import partial

def compute_metrics(pred, tokenizer):
    """Compute WER during training evaluation."""
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # Replace -100 padding with pad_token_id for decoding
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    # Normalize for WER
    pred_str = [normalize_for_wer(p) for p in pred_str]
    label_str = [normalize_for_wer(l) for l in label_str]

    # Filter empty references
    pairs = [(p, l) for p, l in zip(pred_str, label_str) if l.strip()]
    if not pairs:
        return {"wer": 1.0}

    pred_filtered, label_filtered = zip(*pairs)
    wer = jiwer.wer(list(label_filtered), list(pred_filtered))
    return {"wer": wer * 100}

print("✅ Metrics function ready")

✅ Metrics function ready


## 🚂 Step 10: BASELINE Evaluation (Before Fine-Tuning)

First, let's see how the pretrained Whisper-small performs on Hindi **before** fine-tuning.

In [13]:
from tqdm.auto import tqdm

def evaluate_on_fleurs(model_to_eval, processor_to_eval, test_data, max_samples=None, desc="Evaluating"):
    """Evaluate a model on FLEURS test set, return WER and CER."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model_to_eval = model_to_eval.to(device)
    model_to_eval.eval()
    
    all_refs = []
    all_hyps = []
    failures = []
    
    n = min(max_samples, len(test_data)) if max_samples else len(test_data)
    
    for i in tqdm(range(n), desc=desc):
        item = test_data[i]
        audio = item["audio"]["array"]
        sr = item["audio"]["sampling_rate"]
        ref = normalize_for_wer(item["transcription"])
        
        if not ref.strip():
            continue
        
        try:
            input_features = processor_to_eval.feature_extractor(
                audio, sampling_rate=sr, return_tensors="pt"
            ).input_features.to(device)
            
            with torch.no_grad():
                predicted_ids = model_to_eval.generate(
                    input_features, language=LANGUAGE, task=TASK
                )
            
            hyp = processor_to_eval.tokenizer.batch_decode(
                predicted_ids, skip_special_tokens=True
            )[0]
            hyp = normalize_for_wer(hyp)
            
            all_refs.append(ref)
            all_hyps.append(hyp)
            
            sample_wer = jiwer.wer(ref, hyp) if hyp.strip() else 1.0
            if sample_wer > 0.5:
                failures.append({"idx": i, "ref": ref, "hyp": hyp, "wer": f"{sample_wer*100:.1f}%"})
        except Exception as e:
            print(f"Error on sample {i}: {e}")
    
    overall_wer = jiwer.wer(all_refs, all_hyps) * 100
    overall_cer = jiwer.cer(all_refs, all_hyps) * 100
    
    return {
        "wer": round(overall_wer, 2),
        "cer": round(overall_cer, 2),
        "num_samples": len(all_refs),
        "num_failures": len(failures),
        "top_failures": sorted(failures, key=lambda x: x["wer"], reverse=True)[:10],
    }

print("\n" + "="*60)
print("   BASELINE Evaluation — Whisper-small (pretrained)")
print("="*60)

baseline_results = evaluate_on_fleurs(model, processor, fleurs_test, max_samples=50, desc="Baseline")

print(f"\n📊 BASELINE Results:")
print(f"   WER: {baseline_results['wer']}%")
print(f"   CER: {baseline_results['cer']}%")
print(f"   Samples: {baseline_results['num_samples']}")
print(f"   High-error cases: {baseline_results['num_failures']}")


   BASELINE Evaluation — Whisper-small (pretrained)


Baseline:   0%|          | 0/50 [00:00<?, ?it/s]

You have passed task=transcribe, but also have set `forced_decoder_ids` to [[1, None], [2, 50359]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of task=transcribe.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



📊 BASELINE Results:
   WER: 81.32%
   CER: 58.03%
   Samples: 50
   High-error cases: 15


## 🏋️ Step 11: Fine-Tune Whisper on Hindi

Key training decisions:
- **LR = 1e-5**: Prevents catastrophic forgetting of pretrained English knowledge
- **Gradient accumulation = 2**: Effective batch size = 16 on limited Colab GPU
- **FP16**: 2x speedup on T4/V100
- **forced_decoder_ids**: Ensures Hindi decoding at every step

In [14]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=TRAINING_CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=TRAINING_CONFIG["per_device_eval_batch_size"],
    gradient_accumulation_steps=TRAINING_CONFIG["gradient_accumulation_steps"],
    learning_rate=TRAINING_CONFIG["learning_rate"],
    warmup_steps=TRAINING_CONFIG["warmup_steps"],
    num_train_epochs=TRAINING_CONFIG["num_train_epochs"],
    fp16=TRAINING_CONFIG["fp16"] and torch.cuda.is_available(),
    eval_strategy="steps",
    eval_steps=TRAINING_CONFIG["eval_steps"],
    save_strategy="steps",
    save_steps=TRAINING_CONFIG["save_steps"],
    logging_steps=TRAINING_CONFIG["logging_steps"],
    save_total_limit=TRAINING_CONFIG["save_total_limit"],
    load_best_model_at_end=TRAINING_CONFIG["load_best_model_at_end"],
    metric_for_best_model=TRAINING_CONFIG["metric_for_best_model"],
    greater_is_better=TRAINING_CONFIG["greater_is_better"],
    predict_with_generate=True,
    generation_max_length=225,
    report_to=["tensorboard"],
    push_to_hub=False,
    remove_unused_columns=False,
    label_names=["labels"],
    dataloader_num_workers=2,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=partial(compute_metrics, tokenizer=processor.tokenizer),
    tokenizer=processor.feature_extractor,
)

print("🏋️ Starting fine-tuning...")
print(f"   Training samples: {len(train_dataset)}")
print(f"   Eval samples: {len(eval_dataset)}")
print(f"   Epochs: {TRAINING_CONFIG['num_train_epochs']}")
print(f"   Effective batch size: {TRAINING_CONFIG['per_device_train_batch_size'] * TRAINING_CONFIG['gradient_accumulation_steps']}")

🏋️ Starting fine-tuning...
   Training samples: 2120
   Eval samples: 239
   Epochs: 3
   Effective batch size: 16


In [15]:
# ── TRAIN! ──────────────────────────────────────────────────
train_result = trainer.train()

print("\n" + "="*60)
print("   ✅ TRAINING COMPLETE")
print("="*60)
print(f"   Total steps: {train_result.global_step}")
print(f"   Training loss: {train_result.training_loss:.4f}")

Step,Training Loss,Validation Loss


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}



   ✅ TRAINING COMPLETE
   Total steps: 396
   Training loss: 0.5104


## 💾 Step 12: Save Fine-Tuned Model

In [16]:
FINAL_MODEL_PATH = os.path.join(OUTPUT_DIR, "final_model")

model.save_pretrained(FINAL_MODEL_PATH)
processor.save_pretrained(FINAL_MODEL_PATH)

print(f"✅ Model saved to {FINAL_MODEL_PATH}")
print(f"   Size: {sum(os.path.getsize(os.path.join(FINAL_MODEL_PATH, f)) for f in os.listdir(FINAL_MODEL_PATH)) / 1e6:.1f} MB")

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}


✅ Model saved to ./whisper_hindi_finetuned/final_model
   Size: 968.9 MB


## 📊 Step 13: FINE-TUNED Evaluation (After Training)

Now let's evaluate the fine-tuned model and compare with baseline.

In [17]:
print("\n" + "="*60)
print("   FINE-TUNED Evaluation — Whisper-small (Hindi)")
print("="*60)

finetuned_results = evaluate_on_fleurs(model, processor, fleurs_test, max_samples=50, desc="Fine-tuned")

print(f"\n📊 FINE-TUNED Results:")
print(f"   WER: {finetuned_results['wer']}%")
print(f"   CER: {finetuned_results['cer']}%")
print(f"   Samples: {finetuned_results['num_samples']}")


   FINE-TUNED Evaluation — Whisper-small (Hindi)


Fine-tuned:   0%|          | 0/50 [00:00<?, ?it/s]


📊 FINE-TUNED Results:
   WER: 23.5%
   CER: 11.29%
   Samples: 50


## 📋 Step 14: Results Comparison Table

In [18]:
import json

# ── Comparison Table ──────────────────────────────────────────
print("\n" + "="*70)
print(f"{'Model':<35} {'WER (%)':<12} {'CER (%)':<12} {'Samples':<10}")
print("-"*70)
print(f"{'Whisper-small (Baseline)':<35} {baseline_results['wer']:<12.2f} {baseline_results['cer']:<12.2f} {baseline_results['num_samples']:<10}")
print(f"{'Whisper-small (Fine-tuned)':<35} {finetuned_results['wer']:<12.2f} {finetuned_results['cer']:<12.2f} {finetuned_results['num_samples']:<10}")
print("="*70)

wer_delta = finetuned_results['wer'] - baseline_results['wer']
print(f"\nΔ WER: {wer_delta:+.2f}% {'✅ IMPROVED' if wer_delta < 0 else '⚠️ Regressed' if wer_delta > 0 else '─ No change'}")

# ── Save results to JSON ──────────────────────────────────────
all_results = {
    "baseline": baseline_results,
    "finetuned": finetuned_results,
    "delta_wer": wer_delta,
}
with open(os.path.join(OUTPUT_DIR, "wer_results.json"), "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

# ── Markdown table ────────────────────────────────────────────
md_table = f"""# WER Evaluation Results — FLEURS Hindi

| Model | WER (%) | CER (%) | Samples |
|-------|---------|---------|---------|
| Whisper-small (Baseline) | {baseline_results['wer']:.2f} | {baseline_results['cer']:.2f} | {baseline_results['num_samples']} |
| Whisper-small (Fine-tuned) | {finetuned_results['wer']:.2f} | {finetuned_results['cer']:.2f} | {finetuned_results['num_samples']} |

**Δ WER: {wer_delta:+.2f}%**
"""
with open(os.path.join(OUTPUT_DIR, "wer_results.md"), "w", encoding="utf-8") as f:
    f.write(md_table)

print(f"\n📁 Results saved to {OUTPUT_DIR}/")


Model                               WER (%)      CER (%)      Samples   
----------------------------------------------------------------------
Whisper-small (Baseline)            81.32        58.03        50        
Whisper-small (Fine-tuned)          23.50        11.29        50        

Δ WER: -57.82% ✅ IMPROVED

📁 Results saved to ./whisper_hindi_finetuned/


## 🔍 Step 15: Failure Analysis — Top Error Cases

In [19]:
print("\n📋 TOP FAILURE CASES (Fine-tuned model, WER > 50%)\n")
for i, fail in enumerate(finetuned_results.get('top_failures', [])[:5], 1):
    print(f"  Case {i} (WER: {fail['wer']})")
    print(f"    REF: {fail['ref']}")
    print(f"    HYP: {fail['hyp']}")
    print()

if not finetuned_results.get('top_failures'):
    print("  ✅ No high-error cases (all samples < 50% WER)!")


📋 TOP FAILURE CASES (Fine-tuned model, WER > 50%)

  Case 1 (WER: 79.2%)
    REF: कवल द हफत म अमरकय और फर फरच बल न दकषण फरस क मकत कर दय थ और जरमन क ओर बढ रह थ
    HYP: कवल द हबट म अमरकय और फरटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटटट

  Case 2 (WER: 55.2%)
    REF: उनक सगपर क उपपरधन मतर वग कन सग न सवगत कय और उनहन सगपर क परधनमतर ल सएन लग क सथ वयपर और आतकवद क ममल पर बतचत क
    HYP: उनक सगपटय उपपरधन मतर व कनसग न सवगट गय और उनहन सगपटय उपपरधन मतर ल सयन ल क सथदव वयपर और आतग कत क ममल पर बचत क

  Case 3 (WER: 112.5%)
    REF: उनहन अफवह क रजनतक बकवस और मरखतपरण कह
    HYP: उनहन न खओ क रजन तक बपवस और मक ठपम क ह



## 🎤 Step 16: Quick Inference Demo

Test the fine-tuned model on a few samples interactively.

In [20]:
import IPython.display as ipd
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

# Pick 3 random test samples
np.random.seed(42)
demo_indices = np.random.choice(len(fleurs_test), size=3, replace=False)

for idx in demo_indices:
    item = fleurs_test[int(idx)]
    audio = item["audio"]["array"]
    sr = item["audio"]["sampling_rate"]
    ref = item["transcription"]
    
    print(f"\n{'─'*60}")
    print(f"Sample {idx}:")
    print(f"  Reference: {ref}")
    
    # Play audio
    ipd.display(ipd.Audio(audio, rate=sr))
    
    # Transcribe
    inputs = processor.feature_extractor(
        audio, sampling_rate=sr, return_tensors="pt"
    ).input_features.to(device)
    
    with torch.no_grad():
        ids = model.generate(inputs, language=LANGUAGE, task=TASK)
    
    hyp = processor.tokenizer.batch_decode(ids, skip_special_tokens=True)[0]
    print(f"  Predicted: {hyp}")
    
    wer_val = jiwer.wer(normalize_for_wer(ref), normalize_for_wer(hyp))
    print(f"  WER: {wer_val*100:.1f}%")


────────────────────────────────────────────────────────────
Sample 321:
  Reference: हालांकि यह सच नहीं है हालांकि दस्तावेज़ के पीछे कुछ लिखा हुआ है लेकिन यह कोई ख़ज़ाने का नक्शा नहीं है


  Predicted: हालांकि यह सच नहीं है हालांकि दस्तवेश के पीछे कुछ लिखा हुआ है लेकिन यह कोई खज़ाने का नक्षा नहीं है
  WER: 9.5%

────────────────────────────────────────────────────────────
Sample 324:
  Reference: फ़ोटोग्राफ़रों ने बाद में एक वृद्ध महिला की जगह ले ली क्योंकि उसे शौचालय जाना ज़रूरी था मेंडोज़ा को गोली मार दी गई थी


  Predicted: फ़ोटोग्राफ़रों ने बाद में एकप्रिध्ध महिला की जगह ले ली क्योंकि उसे शौचालय जाना ज़रूरी था मिंडोज़ा को घोली मार दी गई थी
  WER: 12.5%

────────────────────────────────────────────────────────────
Sample 388:
  Reference: सभ्यता शब्द लैटिन सिविलिस से आया है जिसका अर्थ है नागरिक लैटिन सिविस से संबंधित है जिसका अर्थ है नागरिक और सिवितास जिसका अर्थ है शहर या शहर-राज्य और वह एक तरह से समाज के आकार को भी परिभाषित करता है


  Predicted: सभ्यताशप्र लैटन सिविटस से आया है जिसका अर्थ है नागरिक लैटन सिविटस से संबंधित है जिसका अर्थ है नागरिक और सिवितास जिसका अर्थ है शहर या शहर राज्य और वह एक तरह से समाच के आकार को भी परिभाषित करता है
  WER: 17.1%


## 📥 Step 17: Download Model (Optional)

Download the fine-tuned model to use locally.

In [21]:
# Zip the model for easy download
import shutil
zip_path = shutil.make_archive("whisper_hindi_finetuned", "zip", OUTPUT_DIR)
print(f"✅ Model zipped: {zip_path}")
print(f"   Download from the file browser on the left (📁)")

# If using Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# shutil.copy(zip_path, '/content/drive/MyDrive/whisper_hindi_finetuned.zip')
# print("✅ Model copied to Google Drive")

✅ Model zipped: /content/whisper_hindi_finetuned.zip
   Download from the file browser on the left (📁)


---
## 📝 Summary

| Step | Done |
|------|------|
| Install deps | ✅ |
| GPU verified | ✅ |
| FLEURS loaded | ✅ |
| Baseline WER | ✅ |
| Fine-tuning | ✅ |
| Fine-tuned WER | ✅ |
| Comparison table | ✅ |
| Model saved | ✅ |